# Match Maker

## Author: Hayden Mann
### Some code based off of code provided from Dr. Kelsey Bisson (will be noted in the code)

1. Install and Imports
2. Configure Paramters
3. IMPORT AUXILARY FUNCTIONS
4. Prepare for For Loop
5. For Loop

# WARNING. WILL TAKE A FEW HOURS WITH CURRENT PARAMETERS. RUN LOCALLY IF POSSIBLE.
## Can downscale aoi/timing below to make for a quicker run to test the code out first

## Install and Imports

In [ ]:
import os
import sys
import glob
import shutil
import warnings
from pathlib import Path
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

warnings.filterwarnings("ignore")


import numpy as np
import pandas as pd


import geopandas as gpd
import rasterio
import rioxarray as rxr
from rasterio.enums import Resampling
from rasterio.features import rasterize
from shapely.geometry import LineString, Polygon
from pyproj import Transformer
from scipy.spatial import cKDTree
from scipy.stats import skew, kurtosis, pearsonr


import xarray as xr
import earthaccess as ea
from sliderule import icesat2, sliderule
from harmony import (
    BBox,
    CapabilitiesRequest,
    Client,
    Collection,
    JobsRequest,
    LinkType,
    Request,
)


import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.dates as mdates
import matplotlib.patheffects as pe
import contextily as cx
import seaborn as sns
from matplotlib.colors import LogNorm, TwoSlopeNorm
from matplotlib.ticker import FuncFormatter, StrMethodFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes


import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


from aux_fx_process import (
    grid_data,
    add_lon_lat,
    get_photons,
    water_mask,
    water_surface_retrieval_and_binning,
    get_segment_vars,
    get_PACE_time_windows,
    get_PACE_data,
    gridded_chl,
    get_segment_centers,
    match_chl_to_segments,
)
from aux_fx_plot import ATLAS_overlays_TX, make_scatter_plot


ea.login()
# ea.login(strategy="interactive", persist=True)

## Configure Study Area + Photon-Input Parameters

In [ ]:
"""" 

Configure Study Area + Photon-Input Parameters

"""

# For Type A
THRESHOLD_LIMIT = 200

# Initialize sliderule
sliderule.init("slideruleearth.io", verbose=True)
pd.set_option("display.max_columns", 80)

# Area along Texas coast
LAT_MIN = 25.911
LAT_MAX = 30.192
LON_MIN = -97.546
LON_MAX = -93.067

# area of interest
aoi = [
        {"lat": LAT_MIN, "lon": LON_MIN},
        {"lat": LAT_MIN, "lon": LON_MAX},
        {"lat": LAT_MAX, "lon": LON_MAX},
        {"lat": LAT_MAX, "lon": LON_MIN},
        {"lat": LAT_MIN, "lon": LON_MIN},
    ]


resources = None

# example base_parms
base_parms = {
    "poly": aoi,
    "t0": "2024-05-01T00:00:00Z",
    "t1": "2026-05-01T23:59:59Z",
    "cnf": 1,
    "srt": 1,
    "atl03_ph_fields": ["pce_mframe_cnt"],
    "atl03_geo_fields": ["solar_elevation"],
}   

POLY = aoi
CNF = 1
SRT = 1
PH_FIELDS = ["pce_mframe_cnt"]
GEO_FIELDS = ["solar_elevation"]

harmony_client = Client(token=ea.get_edl_token())
capabilities_request = CapabilitiesRequest(short_name="PACE_OCI_L2_BGC")
capabilities = harmony_client.submit(capabilities_request)
print(capabilities)

# Load in Shapefile: https://www.arcgis.com/home/item.html?id=595533fecdb0472db4b4b8e3ca8d9e42#overview

LAND = gpd.read_file("ne_10m_land.shp")  

# For temporary data storarge
PACE_ROOT = "DATA/PACE_l2_Texas/"
os.makedirs(PACE_ROOT, exist_ok=True)

# ATLAS SEGMENT BIN PARAMETER
SEGMENT_LENGTH = 1000

# Matchup Point Parameters
TOLERANCE_DEG = 0.02
TOLERANCE_TIME = "12h"


## For-loop parameters

In [ ]:
# Start date for data download + matchups
t0 = "2024-05-01"
t0_object = datetime.strptime(t0, "%Y-%m-%d")

# End date for data download + matchups
t1 = "2026-05-01"
t1_object = datetime.strptime(t1, "%Y-%m-%d")

current = t0_object

# Timestep in the for loop
TIMEDELT = 4

## Matchup For-Loop
# WARNING. WILL TAKE A FEW HOURS. RUN LOCALLY

In [ ]:

""" IMPORT FX """
# from aux_fx_process import add_lon_lat
from aux_fx_process import get_photons
from aux_fx_process import water_mask
from aux_fx_process import water_surface_retrieval_and_binning
from aux_fx_process import get_segment_vars
from aux_fx_process import get_PACE_time_windows
from aux_fx_process import get_PACE_data
from aux_fx_process import gridded_chl
# from aux_fx_process import get_segment_centers
from aux_fx_process import match_chl_to_segments


""" START LOOP """
while current < t1_object:

    """ Set up window dates """

    window_start = current
    window_start_str = window_start.strftime("%Y-%m-%d")
    window_end = current + timedelta(days=TIMEDELT)
    window_end_str = window_end.strftime("%Y-%m-%d")
    print(window_start.strftime("%Y-%m-%d"),
        window_end.strftime("%Y-%m-%d"))
    


    """ Get photons """

    photons_xy = get_photons(window_start_str, window_end_str, aoi)
    if photons_xy is None:
        print(f"No photons | OR SLIDERULE FAIL for {window_start_str} - {window_end_str}, skipping window.")
        current += timedelta(days=TIMEDELT)
        continue

    

    """ Mask Data over water """

    photons_xy_ocean = water_mask(photons_xy, LAND)



    """ Water Surface Retrieval """
    """ 
    
    Credit: Dr. Kelsey Bisson & Eidam, E. F., Bisson, K., Wang, C., Walker, C., & Gibbons, A. (2024). 
    ICESat-2 and ocean particulates: A roadmap for calculating Kd from space-based lidar photon profiles.
    Remote Sensing of Environment, 311, 114222.
      
      
    Adapted by current author to retrieve water height for each individual ground track / beam, to
    account for variations in sea surface height.
    """

    water_surface, below_surface = water_surface_retrieval_and_binning(photons_xy_ocean, SEGMENT_LENGTH)
    if len(below_surface) == 0:
        print(f"No subsurface photons for {window_start_str} - {window_end_str}, skipping window.")
        current += timedelta(days=TIMEDELT)
        continue



    """ Segment Variable Retrieval """

    segment_vars = get_segment_vars(below_surface, water_surface, THRESHOLD_LIMIT)
    if len(segment_vars) == 0:
            print(f"No segments passed THRESHOLD_LIMIT for {window_start_str} - {window_end_str}, skipping window.")
            current += timedelta(days=4)
            continue



    """ Get Time Windows for PACE Data """

    merged_windows_df = get_PACE_time_windows(segment_vars)



    """ Get PACE Data (uncomment out if already done once)
                returns directory for this PACE window data too """

    pace_window_dir = get_PACE_data(merged_windows_df, window_start_str, window_end_str, LON_MIN, LAT_MIN, LON_MAX, LAT_MAX, harmony_client, PACE_ROOT)



    """ Form a gridded geodataframe of PACE chl """

    chl_a_gdf = gridded_chl(pace_window_dir)
    if chl_a_gdf is None:
        print(f"No usable PACE granules for {window_start_str} - {window_end_str}, skipping window.")
        shutil.rmtree(pace_window_dir, ignore_errors=True)
        current += timedelta(days=TIMEDELT)
        continue



    """ Match PACE Chl to ATLAS segments: """ 

    tolerance_deg = TOLERANCE_DEG
    tolerance_time = TOLERANCE_TIME
    matchup_df = match_chl_to_segments(segment_vars, chl_a_gdf, tolerance_deg, tolerance_time)
    matchup_df['chl_a'].describe()



    """  Filter the matchup dataframe to only keep rows where PACE chl_a is NOT NaN """ 

    matchup_df_filtered = matchup_df[matchup_df['chl_a'].notna()]



    """ Print the descriptive statistics """ 

    print(matchup_df_filtered['chl_a'].describe())
    matchup_df
    print(matchup_df_filtered['lidar_date'].value_counts().head())
    matchup_df_filtered



    """ SAVE MATCHUPS """ 

    os.makedirs("MATCHUPS/GULF_DATA_INPACEGAPS", exist_ok=True)  
    os.makedirs("MATCHUPS/GULF_DATA_FILTERED", exist_ok=True) 
    matchup_df.to_parquet("MATCHUPS/GULF_DATA_INPACEGAPS/data._" + window_start_str + "_" + window_end_str + "matchups_GULF_INPASSIVEGAPS")
    matchup_df_filtered.to_parquet("MATCHUPS/GULF_DATA_FILTERED/data._" + window_start_str + "_" + window_end_str + "matchups_GULF")



    """ deletes PACE data now that Matchups downloaded (toggle off to save space) """ 
    
    shutil.rmtree(pace_window_dir, ignore_errors=True)
    print(f"Cleaned up {pace_window_dir}")    



    """ INCREMENT LOOP """
    current += timedelta(days=TIMEDELT)

# Begin Overlay Plots

In [ ]:
""" 

Import Matchups Data 

"""

# Read the matchup df collections
    # filtered: only where PACE chl is not NaN
df_filtered_raw = pd.concat(
    [
        pd.read_parquet(f)
        for f in glob.glob("MATCHUPS/GULF_DATA_FILTERED/data._*")
    ],
    ignore_index=True,
)
df_raw = pd.concat(
    [pd.read_parquet(f) for f in glob.glob("MATCHUPS/GULF_DATA_INPACEGAPS/data._*")],
    ignore_index=True,
)

# Extract standard string dates
df_filtered_raw["date"] = pd.to_datetime(df_filtered_raw["time"]).dt.date
df_raw["date"] = pd.to_datetime(df_raw["time"]).dt.date

# Convert the WKB binary column (saved with the Parquet above) back into actual LineStrings
df_filtered_raw["geo"] = gpd.GeoSeries.from_wkb(df_filtered_raw["geo"])
df_raw["geo"] = gpd.GeoSeries.from_wkb(df_raw["geo"])

# Convert to proper GeoDataFrames
matchup_df_filtered = gpd.GeoDataFrame(
    df_filtered_raw, geometry="geo", crs="EPSG:4326"
)
matchup_df = gpd.GeoDataFrame(df_raw, geometry="geo", crs="EPSG:4326")

print("Filtered Matchup Date Counts:")
print(matchup_df_filtered["date"].value_counts().head())

In [ ]:
""" SELECT DATE OF CHOICE"""
date_array = ["2025-08-12"]

In [ ]:
"""

Get PACE l2m data For the day

"""

# Login to ea again, in case only the bottom section is to be run
auth = ea.login()
harmony_client = Client(token=ea.get_edl_token())
capabilities_request = CapabilitiesRequest(short_name="PACE_OCI_L2_BGC")
capabilities = harmony_client.submit(capabilities_request)
print(capabilities)

# Retrieve lidar from a single date
matchup_df_day = matchup_df[
    pd.to_datetime(matchup_df['date']).dt.strftime("%Y-%m-%d") == date_array[0]
]

# Timestamps. should adjust pd.TimeDelta to match your threshold amount
start = matchup_df_day['lidar_date'].min() - pd.Timedelta(hours=12)
end = matchup_df_day['lidar_date'].max() + pd.Timedelta(hours=12)

# Form request list for EarthData
requests = []
req = Request(
    collection=Collection(id="PACE_OCI_L2_BGC"),
    spatial=BBox(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX),
    temporal={
        "start": start,
        "stop": end
    },
    variables=["geophysical_data/chlor_a"],
    labels={}
)
requests.append(req)


# Submit requests to harmony/EarthData
jobs = []
failed = []

for r in requests:
    try:
        job = harmony_client.submit(r)
        jobs.append(job)
    except Exception as e:
        failed.append((r, str(e)))

print(str(len(failed)) + " out of " + str(len(requests)) + " time windows (+/- 12 hrs) did not have PACE flyover")


# Set output for temporary PACE data
output_dir = "DATA/PACE_l2_Texas"
os.makedirs(output_dir, exist_ok=True)

# Track filenames
downloaded_paths = []
for job_id in jobs:
    try:
        harmony_client.wait_for_processing(job_id, show_progress=True)
        futures = harmony_client.download_all(job_id, directory=output_dir, overwrite=True)

        for future in futures:
            filename = future.result()   # blocks until this file finishes downloading
            downloaded_paths.append(filename)

    except Exception as e:
        print(f"Job {job_id} failed:", e)

print(f"Downloaded {len(downloaded_paths)} files")


# get rid of repeat values in case already donwloaded
seen = set()
unique_files = []

for path in downloaded_paths:
    name = Path(path).name

    if name not in seen:
        seen.add(name)
        unique_files.append(path)


In [ ]:
from aux_fx_plot import ATLAS_overlays_TX
os.makedirs("FINAL_TX_FIGS", exist_ok=True)

# plot resolution
resolution = (0.015, 0.015)

# Set parameters
LAT_MIN = 25.911
LAT_MAX = 30.192
LON_MIN = -97.546
LON_MAX = -93.067
pLAT_MIN = LAT_MIN 
pLAT_MAX = LAT_MAX
pLON_MIN = LON_MIN
pLON_MAX = LON_MAX
params = (pLAT_MIN, pLAT_MAX, pLON_MIN, pLON_MAX, resolution, date_array)




""" PLOTS """


"""R_sw"""

variable = "R_sw"
label = "R$_{sw}$"
norm_col = LogNorm(vmin=0.01, vmax=1000)
filename = "FINAL_TX_FIGS/RSW_NC.png"

ATLAS_overlays_TX(matchup_df, variable, label, norm_col, filename, params)


""" Skewness """

variable = "skewness"
label = "Skewness"
filename = "FINAL_TX_FIGS/Skew_NC.png"
vmin = matchup_df_day[variable].quantile(0.05)
vmax = matchup_df_day[variable].quantile(0.95)
norm_col = colors.Normalize(vmin=vmin, vmax=vmax, clip=True)

ATLAS_overlays_TX(matchup_df, variable, label, norm_col, filename, params)


""" Kurtosis """

variable = "kurtosis"
label = "Kurtosis"
filename = "FINAL_TX_FIGS/Kurt_NC.png"
vmin = matchup_df_day[variable].quantile(0.05)
vmax = matchup_df_day[variable].quantile(0.95)
norm_col = colors.Normalize(vmin=vmin, vmax=vmax, clip=True)

ATLAS_overlays_TX(matchup_df, variable, label, norm_col, filename, params)


"""N_sub"""

variable = "N_subsurface"
label = "N$_{subsurface}$"
vmin = matchup_df_day[variable].quantile(0.05)
vmax = matchup_df_day[variable].quantile(0.95)
norm_col = colors.Normalize(vmin=vmin, vmax=vmax, clip=True)
filename = "FINAL_TX_FIGS/Nsub_NC.png"

ATLAS_overlays_TX(matchup_df, variable, label, norm_col, filename, params)

## References

Eidam, E. F., Bisson, K., Wang, C., Walker, C., & Gibbons, A. (2024). ICESat-2 and ocean particulates: A roadmap for calculating Kd from space-based lidar photon profiles. Remote Sensing of Environment, 311, 114222.